# COMP34812 NLI Shared Task: Demo Code (Inference)
**Group Number:** CW47
**Track:** A. Natural Language Inference (NLI)
**Solution** Category C - Deep Learning (Transformer)
**Group Members:**
1. Munir Emre Tanats - 10959764

## Instructions for the Marker
This notebook runs our trained models in "Inference Mode" on the hidden test dataset.
1. Please ensure you have downloaded our trained model folder (`modernbert-large-rte-final`) from the OneDrive link provided in the `README`.
2. Upload the model folder and your hidden `test.csv` to this environment.
3. Update the `test_df` and `model_dir` variables in the code below to point to your uploaded files.
4. Run all cells to generate the prediction outputs.

---
## Solution 1: Transformer-Based Deep Learning (Category C)
**Model:** Fine-tuned `answerdotai/ModernBERT-Large`

This cell loads our fine-tuned ModernBERT-Large model from the specified directory, processes the premise-hypothesis pairs from the test file, and outputs the binary predictions to a CSV file.

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score, accuracy_score

model_dir = "./final_model_versions/ModernDebertaNew"

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.to(device).eval()

test_df = pd.read_csv("./data/dev.csv")
test_df = test_df.dropna(subset=["premise", "hypothesis", "label"]).copy()

label_mapping = {
    "NOT_ENTAILMENT": 0, "ENTAILMENT": 1,
    "0": 0, "1": 1, "0.0": 0, "1.0": 1,
    0: 0, 1: 1, 0.0: 0, 1.0: 1
}
test_df["label"] = test_df["label"].map(label_mapping).astype(int)
true_labels = test_df["label"].values

texts = [f"{p} [SEP] {h}" for p, h in zip(test_df["premise"], test_df["hypothesis"])]
encodings = tokenizer(texts, truncation=True, max_length=256, padding="max_length", return_tensors="pt")

dataset = TensorDataset(encodings["input_ids"], encodings["attention_mask"])
loader = DataLoader(dataset, batch_size=8)

all_logits = []
with torch.no_grad():
    for input_ids, attention_mask in loader:
        outputs = model(input_ids=input_ids.to(device), attention_mask=attention_mask.to(device))
        all_logits.append(outputs.logits.cpu())

logits = torch.cat(all_logits, dim=0)
probabilities = torch.nn.functional.softmax(logits, dim=-1).numpy()
prob_entailment = probabilities[:, 1]

OPTIMAL_THRESHOLD = 0.50
test_predictions = (prob_entailment >= OPTIMAL_THRESHOLD).astype(int)

print("--- Results on dev.csv ---")
print(f"F1 Score: {f1_score(true_labels, test_predictions, average='binary'):.4f}")
print(f"Accuracy: {accuracy_score(true_labels, test_predictions):.4f}")